### Paramiter influance in LLMs
Now, in the last labs we have made our own CNN and we have seen how Hyperparameters influence the performance of our model (even if we ran into overfitting). Now we will se the same thing but with LLMs. We will see how the number of parameters influence the performance of our model.

There are some more important parameters in NLP models:
- **Temperature**: controls "randomness". Low (0.2) = safe/repetitive. High (1.5) = creative/chaotic.
- **Top-k**: only consider the top k most likely next words. Low (5) = focused. High (50) = diverse.
- **Top-p (nucleus sampling)**: consider the smallest set of words whose cumulative probability exceeds p. Low (0.2) = focused. High (0.9) = diverse.
> what is cumulative probability? It is the sum of the probabilities of the words in the set. For example, if we have a set of words with probabilities [0.1, 0.2, 0.3, 0.4], the cumulative probability of the first two words is 0.1 + 0.2 = 0.3, and the cumulative probability of the first three words is 0.1 + 0.2 + 0.3 = 0.6.
- **Max tokens**: limits the length of the generated text.
- **Repetition penalty**: discourages the model from repeating the same words or phrases. Higher values (e.g., 1.2) make repetition less likely.

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained('gpt2')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [75]:
test_poem = [
  	"From fairest creatures we desire increase,",
	"<<BLANK>>", #"That thereby beauty's rose might never die,"
	"But as the riper should by time decease,",
	"<<BLANK>>", #"His tender heir might bear his memory:"
	"But thou contracted to thine own bright eyes,",
	"Feed'st thy light's flame with self-substantial fuel,",
	"<<BLANK>>", #"Making a famine where abundance lies,"
	"<<BLANK>>", #"Thy self thy foe, to thy sweet self too cruel:"
	"Thou that art now the world's fresh ornament,",
	"And only herald to the gaudy spring,",
	"<<BLANK>>", #"Within thine own bud buriest thy content,"
	"And, tender churl, mak'st waste in niggarding:",
	"<<BLANK>>", #"Pity the world, or else this glutton be,"
	"To eat the world's due, by the grave and thee.",
]

In [76]:
def complete_poem(poem, temperature=0.7, repetition_penalty=1.3, max_new_tokens=20):
	context = ""

	for line in poem:
		print(line)
		if line != "<<BLANK>>":
			context += line + "\n"
		else:
			tokenized_context = tokenizer(context, return_tensors="pt")
			# the new token with the predicted text
			generated_token = model.generate(
				**tokenized_context,
				max_new_tokens=max_new_tokens,
				repetition_penalty=repetition_penalty,
				temperature=temperature,
				do_sample=True,
				pad_token_id=tokenizer.eos_token_id
			)
			# we don't want to have the hole text, just the new part
			input_size = tokenized_context['input_ids'].shape[1]
			new_tokens = generated_token[0][input_size:] # just what GPT generated
			decoded_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

			context += decoded_text + '\n'
	return context

for the first pass of the poen we get

context: From fairest creatures we desire increase,

then this is the generated tokens:
```txt
tensor([[ 4863, 37063,   301,  8109,   356,  6227,  2620,    11,   198,    13,
           764, 22135,   357,    16,  2744,  1315,    25,    24,     8,   628,
          1849,     7,    17,   383,   824, 40755,  1547,   642,    25,  1415,
            12,  1314,  1267,   366,  1870,   262,  4453,   531, 12722,   606,
           837,   705,    40,   481,   787,   345]])
```
and the token without the old tokens is (aka. the context):
```txt
tensor([   13,   764, 22135,   357,    16,  2744,  1315,    25,    24,     8,
          628,  1849,     7,    17,   383,   824, 40755,  1547,   642,    25,
         1415,    12,  1314,  1267,   366,  1870,   262,  4453,   531, 12722,
          606,   837,   705,    40,   481,   787,   345])
```
(2 Thessalonians 5:14-15 ) "And the Lord said unto them , 'I will make you

In [77]:
predicted_poems = [complete_poem(test_poem, temperature=i) for i in [0.3, 0.6, 0.9, 1.2, 1.5]]


From fairest creatures we desire increase,
<<BLANK>>
But as the riper should by time decease,
<<BLANK>>
But thou contracted to thine own bright eyes,
Feed'st thy light's flame with self-substantial fuel,
<<BLANK>>
<<BLANK>>
Thou that art now the world's fresh ornament,
And only herald to the gaudy spring,
<<BLANK>>
And, tender churl, mak'st waste in niggarding:
<<BLANK>>
To eat the world's due, by the grave and thee.
From fairest creatures we desire increase,
<<BLANK>>
But as the riper should by time decease,
<<BLANK>>
But thou contracted to thine own bright eyes,
Feed'st thy light's flame with self-substantial fuel,
<<BLANK>>
<<BLANK>>
Thou that art now the world's fresh ornament,
And only herald to the gaudy spring,
<<BLANK>>
And, tender churl, mak'st waste in niggarding:
<<BLANK>>
To eat the world's due, by the grave and thee.
From fairest creatures we desire increase,
<<BLANK>>
But as the riper should by time decease,
<<BLANK>>
But thou contracted to thine own bright eyes,
Feed'st 

In [78]:
for poem in predicted_poems:
	print("-" * 10)
	print(poem)

----------
From fairest creatures we desire increase,
. . ." (2) "And what is the matter with you?" asked he; and they
But as the riper should by time decease,
"I will not deceive thee." And when thou hast done so , let him go out of thy
But thou contracted to thine own bright eyes,
Feed'st thy light's flame with self-substantial fuel,
So that it may shine in all things which are good: for this I have given unto them ;
For if any man be a god or an angel who hath made his mind clear on matters concerning men
Thou that art now the world's fresh ornament,
And only herald to the gaudy spring,
That shall see through every one whomsoever she seeseth her wayward ways : but whosoever
And, tender churl, mak'st waste in niggarding:
Who has ever seen such great works before? But how can ye know whether these were wrought from God
To eat the world's due, by the grave and thee.

----------
From fairest creatures we desire increase,
- but all men despise the less;

 (1) We have not seen a single ex

### Conclusion
We can see that GPT-2 is very religious, and it's not very creative (crazy, right?? _sarcasm_). We can also see that the model is not very good at generating poetry.

In my opinion the best temperature is 1.2 because it is creative but not too chaotic. The worst temperature is 0.3 because it is too safe and repetitive. It only gives us the bible. All of this to say that the peremeter in this case and with this model doesn't change the outcome that much, it's still shite.